# Subtitulado de Imágenes con Atención Visual

## Objetivos de Aprendizaje
1. Aprender a crear un modelo de subtitulado de imágenes (image captioning)
2. Aprender a entrenar y predecir con un modelo de generación de texto.

Los modelos de subtitulado de imágenes toman una imagen como entrada y generan texto. Idealmente, la salida debe describir con precisión los eventos u objetos en la imagen.

En este notebook utilizaremos una arquitectura de codificador-decodificador con un mecanismo de atención, similar a [Show, Attend and Tell](https://arxiv.org/abs/1502.03044).

## Preparación

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import time
from textwrap import wrap
import keras
import matplotlib.pylab as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from keras import Input
from keras.layers import (
    GRU, Add, Attention, Concatenate, Dense, Embedding,
    LayerNormalization, Rescaling, Reshape, Resizing, StringLookup, TextVectorization
)
print(keras.version())

## Lectura y preparación del conjunto de datos

Usaremos el conjunto de datos [COCO captions](https://www.tensorflow.org/datasets/catalog/coco_captions) a través de TensorFlow Datasets.
Utilizaremos un modelo **InceptionResNetV2** pre-entrenado como extractor de características.

In [ ]:
VOCAB_SIZE = 20000
ATTENTION_DIM = 512
WORD_EMBEDDING_DIM = 128

FEATURE_EXTRACTOR = keras.applications.inception_resnet_v2.InceptionResNetV2(
    include_top=False, weights="imagenet"
)
IMG_HEIGHT = 299
IMG_WIDTH = 299
FEATURES_SHAPE = (8, 8, 1536)

resize = Resizing(IMG_HEIGHT, IMG_WIDTH)
rescale = Rescaling(1.0 / 255)

## Preprocesamiento de Texto

Añadimos tokens especiales para representar el inicio (`<start>`) y el fin (`<end>`) de las oraciones.

In [ ]:
def add_start_end_token(data):
    start = keras.ops.convert_to_tensor("<start>")
    end = keras.ops.convert_to_tensor("<end>")
    data["caption"] = tf.strings.join(
        [start, data["caption"], end], separator=" "
    )
    return data

## Modelo

### Codificador de Imagen
Extrae características usando el modelo pre-entrenado y las pasa a una capa densa.

In [ ]:
FEATURE_EXTRACTOR.trainable = False
image_input = Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
image_features = FEATURE_EXTRACTOR(image_input)
x = Reshape((FEATURES_SHAPE[0] * FEATURES_SHAPE[1], FEATURES_SHAPE[2]))(image_features)
encoder_output = Dense(ATTENTION_DIM, activation="relu")(x)
encoder = keras.Model(inputs=image_input, outputs=encoder_output)

### Decodificador de Subtítulos
Utiliza un mecanismo de atención (estilo Luong) para enfocarse en diferentes partes de la imagen mientras genera el texto.

In [ ]:
word_input = Input(shape=(64,), name="words")
embed_x = Embedding(VOCAB_SIZE, ATTENTION_DIM)(word_input)
decoder_gru = GRU(ATTENTION_DIM, return_sequences=True, return_state=True)
gru_output, gru_state = decoder_gru(embed_x)
decoder_attention = Attention()
context_vector = decoder_attention([gru_output, encoder_output])
addition = Add()([gru_output, context_vector])
layer_norm_out = LayerNormalization(axis=-1)(addition)
decoder_output = Dense(VOCAB_SIZE)(layer_norm_out)
decoder = keras.Model(inputs=[word_input, encoder_output], outputs=decoder_output)

## Bucle de Entrenamiento
Entrenamos el modelo combinando el codificador y el decodificador.

In [ ]:
image_caption_train_model = keras.Model(inputs=[image_input, word_input], outputs=decoder_output)
image_caption_train_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
# image_caption_train_model.fit(batched_ds, epochs=1)

## Resumen
Hemos aprendido a construir un modelo de subtitulado de imágenes creando un codificador de imágenes y un decodificador de texto con atención.